In [5]:
# ============================================================
# Memory-efficient cumulative GIF from classified raster
# - high and very high merged
# - imports country boundaries freshly
# - inferno palette
# - visible country boundaries
# - less crowded right-side panel
# ============================================================

from pathlib import Path
import numpy as np
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
import matplotlib as mpl
from PIL import Image
import geopandas as gpd
from shapely.geometry import box

# ------------------------------------------------------------
# Input / output
# ------------------------------------------------------------

raster_fp = "/mnt/eo/EO4Backcasting/_predictions/probability_bins_5classes.tif"

out_dir = Path("/mnt/eo/EO4Backcasting/_figures/continuity_gif")
out_dir.mkdir(parents=True, exist_ok=True)

gif_fp = out_dir / "continuity_classes_cumulative_high_veryhigh_merged.gif"

# ------------------------------------------------------------
# Optional: if you have your own countries file, set it here
# Otherwise the code tries to load Natural Earth countries
# ------------------------------------------------------------

countries_fp = None
# countries_fp = "/path/to/your/europe_countries.shp"
# countries_fp = "/path/to/your/europe_countries.gpkg"

# ------------------------------------------------------------
# Class settings
# ------------------------------------------------------------
# Original raster:
# 1 = very low
# 2 = low
# 3 = moderate
# 4 = high
# 5 = very high
#
# New plotting classes:
# 1 = very low
# 2 = low
# 3 = moderate
# 4 = high + very high
# ------------------------------------------------------------

class_labels = {
    1: "Very low",
    2: "Low",
    3: "Moderate",
    4: "High",
}

valid_classes = [1, 2, 3, 4]
add_order = [4, 3, 2, 1]

# Inferno colors:
# low = bright yellow/orange
# high + very high = dark purple/black
inferno_values = {
    1: 0.95,
    2: 0.72,
    3: 0.48,
    4: 0.10,
}

class_info = {}
for cls in valid_classes:
    rgba = mpl.colormaps["inferno"](inferno_values[cls])
    hex_col = "#{:02x}{:02x}{:02x}".format(
        int(rgba[0] * 255),
        int(rgba[1] * 255),
        int(rgba[2] * 255)
    )
    class_info[cls] = {
        "label": class_labels[cls],
        "color": hex_col
    }

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def format_area_ha(area_ha):
    if area_ha >= 1_000_000:
        return f"{area_ha / 1_000_000:.2f} Mha"
    elif area_ha >= 1_000:
        return f"{area_ha / 1_000:.1f} kha"
    else:
        return f"{area_ha:.0f} ha"


def hex_to_rgba01(h, alpha=1.0):
    h = h.lstrip("#")
    return tuple(int(h[j:j+2], 16) / 255 for j in (0, 2, 4)) + (alpha,)


def load_countries():
    """
    Load country boundaries freshly.
    Priority:
    1. user-provided countries_fp
    2. Natural Earth via geodatasets
    3. Natural Earth via cartopy
    4. Natural Earth online URL
    """

    if countries_fp is not None:
        print("Loading countries from:", countries_fp)
        return gpd.read_file(countries_fp)

    # Try geodatasets
    try:
        import geodatasets
        ne_path = geodatasets.get_path("naturalearth.countries")
        print("Loading countries from geodatasets:", ne_path)
        return gpd.read_file(ne_path)
    except Exception as e:
        print("geodatasets loading failed:", e)

    # Try cartopy
    try:
        import cartopy.io.shapereader as shpreader
        ne_path = shpreader.natural_earth(
            resolution="50m",
            category="cultural",
            name="admin_0_countries"
        )
        print("Loading countries from cartopy Natural Earth:", ne_path)
        return gpd.read_file(ne_path)
    except Exception as e:
        print("cartopy loading failed:", e)

    # Try online Natural Earth
    try:
        ne_url = (
            "https://naturalearth.s3.amazonaws.com/"
            "50m_cultural/ne_50m_admin_0_countries.zip"
        )
        print("Loading countries from Natural Earth URL.")
        return gpd.read_file(ne_url)
    except Exception as e:
        print("Online Natural Earth loading failed:", e)

    raise RuntimeError(
        "Could not load country boundaries. "
        "Please set countries_fp to a local shapefile or geopackage."
    )


# ------------------------------------------------------------
# Read raster metadata and downsampled raster for plotting
# ------------------------------------------------------------

with rasterio.open(raster_fp) as src:
    crs = src.crs
    nodata = src.nodata
    transform = src.transform
    bounds = src.bounds
    width = src.width
    height = src.height

    if crs is None:
        raise ValueError("Raster has no CRS.")

    if crs.is_geographic:
        raise ValueError(
            "Raster is in geographic CRS. "
            "Please reproject to a projected/equal-area CRS before area calculation."
        )

    max_dim = 1800
    scale = min(max_dim / width, max_dim / height, 1.0)

    out_height = int(height * scale)
    out_width = int(width * scale)

    print("Original raster size:", width, "x", height)
    print("Plot raster size:", out_width, "x", out_height)
    print("Raster CRS:", crs)
    print("Raster bounds:", bounds)

    arr_plot = src.read(
        1,
        out_shape=(out_height, out_width),
        resampling=Resampling.nearest
    )

    # Merge class 5 into class 4 for plotting only
    arr_plot = arr_plot.copy()
    arr_plot[arr_plot == 5] = 4

    pixel_area_m2 = abs(transform.a * transform.e)
    pixel_area_ha = pixel_area_m2 / 10_000

# ------------------------------------------------------------
# Load and prepare country boundaries
# ------------------------------------------------------------

countries_gdf = load_countries()

print("Countries loaded:", countries_gdf.shape)
print("Countries original CRS:", countries_gdf.crs)

if countries_gdf.crs is None:
    countries_gdf = countries_gdf.set_crs("EPSG:4326")

# Keep European countries if continent attribute exists
if "CONTINENT" in countries_gdf.columns:
    countries_gdf = countries_gdf[countries_gdf["CONTINENT"].isin(["Europe"])].copy()
elif "continent" in countries_gdf.columns:
    countries_gdf = countries_gdf[countries_gdf["continent"].isin(["Europe"])].copy()

# Reproject to raster CRS
countries_gdf = countries_gdf.to_crs(crs)

# Clip/filter to raster extent
raster_box = gpd.GeoDataFrame(
    geometry=[box(bounds.left, bounds.bottom, bounds.right, bounds.top)],
    crs=crs
)

try:
    countries_gdf = gpd.overlay(countries_gdf, raster_box, how="intersection")
except Exception:
    countries_gdf = countries_gdf[countries_gdf.intersects(raster_box.geometry.iloc[0])].copy()

print("Countries after Europe/extent filter:", countries_gdf.shape)
print("Countries projected bounds:", countries_gdf.total_bounds)

# ------------------------------------------------------------
# Blockwise area calculation
# ------------------------------------------------------------

class_counts = {cls: 0 for cls in valid_classes}
total_valid_pixels = 0

with rasterio.open(raster_fp) as src:
    for _, window in src.block_windows(1):
        block = src.read(1, window=window)

        # Merge class 5 into class 4 for area calculation only
        block = block.copy()
        block[block == 5] = 4

        if nodata is not None:
            valid = block != nodata
        else:
            valid = np.ones(block.shape, dtype=bool)

        valid &= np.isin(block, valid_classes)

        total_valid_pixels += int(valid.sum())

        for cls in valid_classes:
            class_counts[cls] += int(np.sum((block == cls) & valid))

class_area_ha = {
    cls: count * pixel_area_ha
    for cls, count in class_counts.items()
}

total_valid_area_ha = total_valid_pixels * pixel_area_ha

print("\nClass areas after merging high + very high:")
for cls in add_order:
    print(class_info[cls]["label"], ":", format_area_ha(class_area_ha[cls]))

print("Total valid area:", format_area_ha(total_valid_area_ha))

# ------------------------------------------------------------
# Prepare plotting masks
# ------------------------------------------------------------

if nodata is not None:
    valid_mask_plot = arr_plot != nodata
else:
    valid_mask_plot = np.ones(arr_plot.shape, dtype=bool)

valid_mask_plot &= np.isin(arr_plot, valid_classes)

plot_color_map = {0: "#f2f2f2"}
for cls, info in class_info.items():
    plot_color_map[cls] = info["color"]

cumulative_sets = []
current = []
for cls in add_order:
    current.append(cls)
    cumulative_sets.append(current.copy())

# ------------------------------------------------------------
# Create frames
# ------------------------------------------------------------

frame_paths = []

for i, included_classes in enumerate(cumulative_sets, start=1):

    frame_arr = np.zeros(arr_plot.shape, dtype=np.uint8)

    keep = valid_mask_plot & np.isin(arr_plot, included_classes)
    frame_arr[keep] = arr_plot[keep].astype(np.uint8)

    rgba = np.zeros((frame_arr.shape[0], frame_arr.shape[1], 4), dtype=np.float32)

    for val, col in plot_color_map.items():
        rgba[frame_arr == val] = hex_to_rgba01(col, alpha=1.0)

    included_area_ha = sum(class_area_ha[c] for c in included_classes)
    included_pct = included_area_ha / total_valid_area_ha * 100

    added_class = included_classes[-1]
    added_label = class_info[added_class]["label"]

    # Larger right panel and more vertical room
    fig = plt.figure(figsize=(13.2, 8.8), dpi=140)

    gs = fig.add_gridspec(
        nrows=1,
        ncols=2,
        width_ratios=[4.9, 2.1],
        wspace=0.05
    )

    ax_map = fig.add_subplot(gs[0, 0])
    ax_side = fig.add_subplot(gs[0, 1])

    # Map
    ax_map.imshow(
        rgba,
        extent=(bounds.left, bounds.right, bounds.bottom, bounds.top),
        origin="upper",
        zorder=1
    )

    # Country boundaries: white underlay + dark line
    if countries_gdf is not None and len(countries_gdf) > 0:
        countries_gdf.boundary.plot(
            ax=ax_map,
            color="white",
            linewidth=0.95,
            zorder=20
        )
        countries_gdf.boundary.plot(
            ax=ax_map,
            color="black",
            linewidth=0.25,
            zorder=21
        )

    ax_map.set_xlim(bounds.left, bounds.right)
    ax_map.set_ylim(bounds.bottom, bounds.top)

    ax_map.set_title(
        f"Cumulative backcast forest continuity classes\nAdded class: {added_label}",
        fontsize=15,
        fontweight="bold",
        pad=12
    )

    ax_map.set_xticks([])
    ax_map.set_yticks([])

    for spine in ax_map.spines.values():
        spine.set_visible(False)

    # --------------------------------------------------------
    # Right side panel
    # --------------------------------------------------------

    ax_side.axis("off")

    ax_side.text(
        0.00, 0.97,
        "Probability class",
        fontsize=14,
        fontweight="bold",
        ha="left",
        va="top",
        transform=ax_side.transAxes
    )

    y0 = 0.88
    dy = 0.08

    for j, cls in enumerate(add_order):
        active = cls in included_classes
        alpha_box = 1.0 if active else 0.18
        alpha_text = 1.0 if active else 0.40

        ax_side.add_patch(
            plt.Rectangle(
                (0.00, y0 - j * dy - 0.024),
                0.075,
                0.038,
                color=class_info[cls]["color"],
                alpha=alpha_box,
                transform=ax_side.transAxes,
                clip_on=False
            )
        )

        ax_side.text(
            0.10,
            y0 - j * dy,
            class_info[cls]["label"],
            fontsize=11,
            ha="left",
            va="center",
            alpha=alpha_text,
            fontweight="bold" if active else "normal",
            transform=ax_side.transAxes
        )

    # Main statistic
    ax_side.text(
        0.00, 0.49,
        "Cumulative area",
        fontsize=14,
        fontweight="bold",
        ha="left",
        va="top",
        transform=ax_side.transAxes
    )

    ax_side.text(
        0.00, 0.405,
        format_area_ha(included_area_ha),
        fontsize=20,
        fontweight="bold",
        ha="left",
        va="top",
        transform=ax_side.transAxes
    )

    ax_side.text(
        0.00, 0.33,
        f"{included_pct:.1f}% of valid forest area",
        fontsize=11,
        ha="left",
        va="top",
        transform=ax_side.transAxes
    )

    # Compact class list
    ax_side.text(
        0.00, 0.24,
        "Included class areas",
        fontsize=12.5,
        fontweight="bold",
        ha="left",
        va="top",
        transform=ax_side.transAxes
    )

    yy = 0.19
    for cls in included_classes:
        ax_side.text(
            0.00,
            yy,
            f"{class_info[cls]['label']}",
            fontsize=10,
            ha="left",
            va="top",
            transform=ax_side.transAxes
        )
        ax_side.text(
            0.58,
            yy,
            format_area_ha(class_area_ha[cls]),
            fontsize=10,
            ha="left",
            va="top",
            transform=ax_side.transAxes
        )
        yy -= 0.05

    fig.subplots_adjust(
        bottom=0.10,
        top=0.92,
        left=0.03,
        right=0.98
    )

    frame_fp = out_dir / f"frame_{i:02d}_high_veryhigh_merged.png"

    fig.savefig(
        frame_fp,
        dpi=140,
        bbox_inches="tight",
        pad_inches=0.45,
        facecolor="white"
    )

    plt.close(fig)

    frame_paths.append(frame_fp)
    print("Saved frame:", frame_fp)

# ------------------------------------------------------------
# Create GIF
# ------------------------------------------------------------

frames = [Image.open(fp).convert("P", palette=Image.ADAPTIVE) for fp in frame_paths]

frames[0].save(
    gif_fp,
    save_all=True,
    append_images=frames[1:],
    duration=1200,
    loop=0
)

for f in frames:
    f.close()

print("\nSaved GIF:", gif_fp)

Original raster size: 130000 x 140000
Plot raster size: 1671 x 1800
Raster CRS: EPSG:3035
Raster bounds: BoundingBox(left=2612197.933345846, bottom=1365550.4228236396, right=6512197.933345846, top=5565550.42282364)
geodatasets loading failed: No module named 'geodatasets'
cartopy loading failed: No module named 'cartopy'
Loading countries from Natural Earth URL.
Countries loaded: (242, 169)
Countries original CRS: EPSG:4326
Countries after Europe/extent filter: (50, 169)
Countries projected bounds: [2636158.70849328 1429726.81340528 6512197.93334585 5565550.42282364]

Class areas after merging high + very high:
High : 16.33 Mha
Moderate : 74.26 Mha
Low : 55.96 Mha
Very low : 8.47 Mha
Total valid area: 155.02 Mha
Saved frame: /mnt/eo/EO4Backcasting/_figures/continuity_gif/frame_01_high_veryhigh_merged.png
Saved frame: /mnt/eo/EO4Backcasting/_figures/continuity_gif/frame_02_high_veryhigh_merged.png
Saved frame: /mnt/eo/EO4Backcasting/_figures/continuity_gif/frame_03_high_veryhigh_merged.